In [ ]:
!pip install adapter-transformers datasets evaluate rouge_score sentencepiece pythainlp attacut
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import evaluate
from tqdm import tqdm
from pythainlp.tokenize import word_tokenize
import pickle
from datasets import Dataset
from tqdm import tqdm
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Inference Evaluate and Test

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("thanathorn/mt5-cpe-kmutt-thai-sentence-sum")
model = T5ForConditionalGeneration.from_pretrained('/content/drive/MyDrive/my-mam-mt5').to("cuda")
model.load_adapter('/content/drive/MyDrive/my-mam-mt5')
model.set_active_adapters('mam_adapter')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Overwriting existing adapter 'mam_adapter'.


In [ ]:
with open('/content/drive/MyDrive/tokenized_data.pkl', 'rb') as f:
    ds = pickle.load(f)

train_num = int(0.6*len(ds['input_ids'])) #133,531
eval_num = int((len(ds['input_ids'])-train_num)/2) #44,511
test_num = len(ds['input_ids'])-(train_num+eval_num) #44,511
print(train_num,eval_num,test_num) #133,531 44,511 44,511
print(train_num+eval_num+test_num) #222,553

# train_dataset = Dataset.from_dict({'input_ids':ds['input_ids'][:train_num],'attention_mask':ds['attention_mask'][:train_num],'labels':ds['labels'][:train_num]})
eval_dataset = Dataset.from_dict({'input_ids':ds['input_ids'][train_num:train_num+eval_num],'attention_mask':ds['attention_mask'][train_num:train_num+eval_num],'labels':ds['labels'][train_num:train_num+eval_num]})
test_dataset = Dataset.from_dict({'input_ids':ds['input_ids'][train_num+eval_num:],'attention_mask':ds['attention_mask'][train_num+eval_num:],'labels':ds['labels'][train_num+eval_num:]})

133531 44511 44511
222553


In [ ]:
# pred eval
df_pred_eval  = pd.DataFrame({'eval_pred':['' for i in range(eval_num)]})
batch_size = 50
for i in tqdm(range(0,eval_num,batch_size)):
  model.eval()
  with torch.no_grad():
    eval_pred=[]
    ids = torch.LongTensor(eval_dataset['input_ids'][i:i+batch_size]).to('cuda')
    mas = torch.LongTensor(eval_dataset['attention_mask'][i:i+batch_size]).to('cuda')
    m = model.generate(input_ids=ids,attention_mask=mas,max_length=1000)
    for j in range(len(m)):
      s = tokenizer.decode(m[j][1:])
      ans = s[:s.find('</s>')]
      eval_pred.append([ans])
    df_pred_eval.iloc[i:i+len(m)] = eval_pred

df_pred_eval.to_csv('/content/drive/MyDrive/df_pred_eval.csv',index=False)

100%|██████████| 891/891 [6:00:14<00:00, 24.26s/it]


In [ ]:
#pred test
df_pred_test  = pd.DataFrame({'test_pred':['' for i in range(test_num)]})
batch_size = 50
for i in tqdm(range(0,test_num,batch_size)):
  model.eval()
  with torch.no_grad():
    test_pred=[]
    ids = torch.LongTensor(test_dataset['input_ids'][i:i+batch_size]).to('cuda')
    mas = torch.LongTensor(test_dataset['attention_mask'][i:i+batch_size]).to('cuda')
    m = model.generate(input_ids=ids,attention_mask=mas,max_length=1000)
    for j in range(len(m)):
      s = tokenizer.decode(m[j][1:])
      ans = s[:s.find('</s>')]
      test_pred.append([ans])
    df_pred_test.iloc[i:i+len(m)] = test_pred

df_pred_test.to_csv('/content/drive/MyDrive/df_pred_test.csv',index=False)

100%|██████████| 891/891 [5:44:10<00:00, 23.18s/it]


In [ ]:
df_pred_eval.tail(10)

In [ ]:
df_pred_test.tail(10)

,test_pred
44501,ช้างศึก ทีมชาติไทย โชว์ฟอร์มสุดโหด บุกมาเอาชนะ...
44502,ช้างเหนือ เชียงใหม่ จัดงานใหญ่ ช้างป่า ช้างป่า...
44503,ชาวบ้านที่ อ.กันทรลักษ์ จ.ศรีสะเกษ ปลูกต้นกล้ว...
44504,เด็กหญิงวัย 60 ปี วัย 60 ปี มาว่าที่บ้านหลังนี...
44505,เจียง อี้-หัว นายกรัฐมนตรีไต้หวัน ประกาศลาออกจ...
44506,นิพิฏฐ์ โพสต์ข้อความในเฟซบุ๊กส่วนตัว ชี้ บิ๊กต...
44507,กองทัพบก ร่วมซ้อมใหญ่พิธีเทิดพระเกียรติพระบาทส...
44508,ชาวบ้านที่ อ.เมือง จ.ตรัง พบต้นกล้วยประหลาดออก...
44509,หนุ่มโคราชซิ่งบิ๊กไบค์ หลุดโค้งภูเสี้ยวชนรถกระ...
44510,นักท่องเที่ยวแห่ชมวิวทิวทัศน์ มุม ขุนเขา แมกไม...


#Cal Score

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/thairath-222_final.csv')
test_df = pd.read_csv('/content/drive/MyDrive/df_pred.csv')
eval_df = pd.read_csv('/content/drive/MyDrive/df_pred_eval.csv')

In [ ]:
num_train = 133531
num_eval = 44511
num_test = 44511

In [ ]:
#cal eval
rouge = evaluate.load('rouge')
result_eval = rouge.compute(predictions=list(eval_df['eval_pred']),
                       references=list(df['summary'].iloc[-(num_test+num_eval):-num_test]),
                       tokenizer=lambda x:word_tokenize(x,engine='attacut'))
result_eval

{'rouge1': 0.4095211818124974,
 'rouge2': 0.20734951837936538,
 'rougeL': 0.34942393091581003,
 'rougeLsum': 0.34962538310634694}

In [ ]:
#cal test
rouge = evaluate.load('rouge')
result_test = rouge.compute(predictions=list(test_df['test_pred']),
                       references=list(df['summary'].iloc[-test_df.shape[0]:]),
                       tokenizer=lambda x:word_tokenize(x,engine='attacut'))
result_test